In [6]:
import numpy as np
import pandas as pd
from google.cloud import bigquery

pd.set_option('display.max_rows', None)

client = bigquery.Client()

## Test 1

In [7]:
query = f"""
SELECT
  id_customer,
  name,
  is_current,
  valid_from,
  valid_to
FROM `bake2home-data-warehouse.gold.dim_customer`
WHERE id_customer = 1
ORDER BY valid_from DESC;
"""

result = client.query(query).to_dataframe()
display(result)

,id_customer,name,is_current,valid_from,valid_to
0,1,Jan Nowak,True,2025-05-22 09:48:35.269474+00:00,9999-12-31 23:59:59+00:00


In [8]:
query = f"""
SELECT
  id_customer,
  name,
  is_current,
  valid_from,
  valid_to
FROM `bake2home-data-warehouse.gold.dim_customer`
WHERE id_customer = 1
ORDER BY valid_from DESC;
"""

result = client.query(query).to_dataframe()
display(result)

,id_customer,name,is_current,valid_from,valid_to
0,1,Jan Nowak - ZMIENIONE DO TESTU,True,2026-06-15 08:26:22.404190+00:00,9999-12-31 23:59:59+00:00
1,1,Jan Nowak,False,2025-05-22 09:48:35.269474+00:00,2026-06-15 08:26:22.404190+00:00


## Test 2

In [9]:
table_lineage = {
    'fact_payment': {'silver': 'stg_payment', 'bronze': 'raw_wallet_Payments'},
    'fact_order_item': {'silver': 'stg_bakery_order_items', 'bronze': 'raw_planner_BakeryOrderItems'},
    'fact_delivery_item': {'silver': 'stg_delivery_items', 'bronze': 'raw_delivery_manager_DeliveryItems'},
    'fact_refund_item': {'silver': 'stg_refund_items', 'bronze': 'raw_wallet_RefundItems'},
    'fact_planner_item': {'silver': 'stg_planner_item', 'bronze': 'raw_planner_PlannerItems'},

    'dim_customer': {'silver': 'stg_customer', 'bronze': 'raw_customer_manager_Customer'},
    'dim_customer_address': {'silver': 'stg_customer_address', 'bronze': 'raw_customer_manager_CustomerAddress'},
    'dim_product': {'silver': 'stg_bakery_product', 'bronze': 'raw_catalog_BakeryProduct'},
    'dim_payment_method': {'silver': 'stg_payment_method', 'bronze': 'raw_wallet_PaymentMethods'},
    'dim_site': {'silver': 'stg_site', 'bronze': 'raw_client_manager_Site'},
    'dim_delivery_man': {'silver': 'stg_delivery_man', 'bronze': 'raw_delivery_account_manager_DeliveryMan'},
    'dim_discount': {'silver': 'stg_discount', 'bronze': 'raw_wallet_Discounts'},
    'dim_location': {'silver': 'stg_location', 'bronze': 'raw_client_manager_Location'},
    'dim_allergen': {'silver': 'stg_allergen', 'bronze': 'raw_catalog_Allergen'},
    'dim_offer': {'silver': 'stg_bakery_offer', 'bronze': 'raw_catalog_BakeryOffer'},
    'dim_competitor': {'silver': 'stg_google_competitors', 'bronze': 'raw_google_competitors'}
}

print(f"Scanning {len(table_lineage)} main data paths. This will take a moment...")
dfs = []

for gold_table, lineage in table_lineage.items():
    silver_table = lineage['silver']
    bronze_table = lineage['bronze']

    targets = [
        ('1_Bronze', 'bronze', bronze_table),
        ('2_Silver', 'silver', silver_table),
        ('3_Gold', 'gold', gold_table)
    ]

    for layer_alias, dataset_name, table_name in targets:
        q = f"SELECT '{gold_table}' AS main_table, '{layer_alias}' AS layer, COUNT(*) AS row_count FROM `{project}.{dataset_name}.{table_name}`"

        try:
            dfs.append(client.query(q).to_dataframe())
        except Exception as e:
            print(f"Error for {dataset_name}.{table_name}: {e}")

if dfs:
    df_raw = pd.concat(dfs, ignore_index=True)
    df_pivot = df_raw.pivot(index='main_table', columns='layer', values='row_count')
    df_pivot.columns = [col.split('_')[1] for col in df_pivot.columns]

    print("\nDATA PATH CONSISTENCY AUDIT")
    display(df_pivot)
else:
    print("No data to display.")

Scanning 16 main data paths. This will take a moment...

DATA PATH CONSISTENCY AUDIT


,Bronze,Silver,Gold
main_table,,,
dim_allergen,15,15,16
dim_competitor,483,483,481
dim_customer,176,176,178
dim_customer_address,544,544,545
dim_delivery_man,6,6,7
dim_discount,21,21,22
dim_location,35,35,36
dim_offer,145,145,146
dim_payment_method,187,187,188


## Test 3

In [11]:
query = f"""
SELECT
  c.id_competitor,
  c.name AS competitor_name,
  s.id_site,
  s.name AS our_site_name,
  ST_DISTANCE(c.geo_point, s.geo_point) AS distance_meters
FROM `bake2home-data-warehouse.gold.dim_competitor` c
CROSS JOIN `bake2home-data-warehouse.gold.dim_site` s
WHERE c.id_competitor != "-999999"
  AND s.id_site != -999999
  AND s.is_current = TRUE
  AND ST_DISTANCE(c.geo_point, s.geo_point) < 30.0
ORDER BY distance_meters ASC;
"""

result = client.query(query).to_dataframe()
display(result)

,id_competitor,competitor_name,id_site,our_site_name,distance_meters
